In [4]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from selenium.common.exceptions import TimeoutException

In [5]:
# open the browser
# browser = webdriver.Chrome(executable_path = path)


In [6]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains
from selenium.common.exceptions import NoSuchElementException
import time

url = 'https://stocktrack.ca/?s=ikea&search=artificial%20plants'
driver = webdriver.Chrome()
driver.get(url)
time.sleep(30)

# Wait for the iframe to load and switch to it
try:
    wait = WebDriverWait(driver, 20)  # Increased timeout
    iframe = wait.until(EC.presence_of_element_located((By.TAG_NAME, 'iframe')))
    driver.switch_to.frame(iframe)
except TimeoutException:
    print("No iframe found or timed out waiting for iframe to load.")
except NoSuchElementException:
    print("No iframe element found.")

# Now wait for the products list to load within the iframe
try:
    products_list = wait.until(EC.presence_of_all_elements_located((By.CLASS_NAME, "dhx_list_item")))

    scraped_products = []
    # Iterate over the product list and extract information
    for product in products_list:
    # Find elements within each product
        image = product.find_element(By.TAG_NAME, 'img').get_attribute('src')
        name = product.find_element(By.TAG_NAME, 'a').text
        link = product.find_element(By.TAG_NAME, 'a').get_attribute('href')
        
        product_info = product.find_elements(By.XPATH, './/td')[1].text
        product_info_lines = product_info.split('\n')
        #second_td_element = product.find_elements(By.TAG_NAME, 'td')[1]
        #product_size = second_td_element.text.strip()  # Extract size information from the second <td> element
        if ":" in product_info_lines[1].strip():
            product_size = "N/A"  # Set a default value if size is not available
            product_sku = product_info_lines[1].split(': ')[1]
            product_price = product_info_lines[2].split(': ')[1]
        else:
            product_size = product_info_lines[1].strip() 
            product_sku = product_info_lines[2].split(': ')[1]
            product_price = product_info_lines[3].split(': ')[1]   
        
        if " " in product_price:
            prices = product_price.split()
            old_price = prices[0]  # First part is old price
            new_price = prices[1]  # Second part is new price
        else:
            old_price = product_price
            new_price = 'N/A'
        product_status_element = product.find_element(By.CLASS_NAME, 'instock')
        product_stat = product_status_element.text
        
        # Store the product data in a dictionary
        product_data = {
            'image_url': image,
            'product_name': name,
            'product_link': link,
            'product_size' : product_size,
            'product_sku': product_sku,
            'product_price_old': old_price,
            'product_price_new' : new_price,
            'product_status': product_stat
        }
        
        # Append the product data to the list
        scraped_products.append(product_data)
        # Output the information
        #print(f"Image: {image}, Name: {name}, Link: {link}, product_size : {product_size}, product_sku : {product_sku}, product_stat : {product_stat}")
except TimeoutException:
    print("Timed out waiting for products to load")

# Print the scraped product data
for product in scraped_products:
    print(product)

div_to_click = driver.find_element(By.XPATH, '//div[@dhx_f_id="1"]')
div_to_click.click()
time.sleep(5)
div_to_click = driver.find_element(By.XPATH, '//div[@dhx_f_id="2"]')
div_to_click.click()
# Close the WebDriver
#driver.quit()





{'image_url': 'https://www.ikea.com/ca/en/images/products/fejka-artificial-potted-plant-indoor-outdoor-bamboo__0748884_pe745273_s5.jpg', 'product_name': 'FEJKA, Artificial potted plant', 'product_link': 'https://www.ikea.com/ca/en/p/fejka-artificial-potted-plant-indoor-outdoor-bamboo-10467804/', 'product_size': '23 cm (9 ")', 'product_sku': '10467804', 'product_price_old': '$69.99', 'product_price_new': 'N/A', 'product_status': 'Available'}
{'image_url': 'https://www.ikea.com/ca/en/images/products/fejka-artificial-plant-with-wall-holder-indoor-outdoor-green-lilac__1184665_pe898020_s5.jpg', 'product_name': 'FEJKA, Artificial plant with wall holder', 'product_link': 'https://www.ikea.com/ca/en/p/fejka-artificial-plant-with-wall-holder-indoor-outdoor-green-lilac-30548625/', 'product_size': 'N/A', 'product_sku': '30548625', 'product_price_old': '$6.99', 'product_price_new': 'N/A', 'product_status': 'Available'}
{'image_url': 'https://www.ikea.com/ca/en/images/products/fejka-artificial-pott